## Setup

Keep [`example_data.py`](example_data.py) and [`privacy_metrics.py`](privacy_metrics.py) beside this notebook.

In [1]:
#| label: setup
from IPython.display import display

from example_data import METRIC_CATEGORICAL_COLUMNS, make_example_views
from privacy_metrics import distance_membership_auc, distance_membership_details

## 1. Metric, inputs, and notation

The **Membership-Inference AUC** is a privacy metric adapted from the source project. We write it as $\mathrm{MIA}_{\mathrm{privacy}}$. It uses a distance-based attack to ask whether training members are systematically closer to the released table than unseen non-members are.

For this privacy interpretation, the datasets have specific roles. In the typical synthetic-data setting, the real dataset $X$ is used to train a generator, and the generator produces the released synthetic dataset $Y$. A row $x\in X$ is therefore a **member** because the generator was trained using it. The independent real holdout $V$ is not used to train the generator, so a row $v\in V$ is a **non-member**. If $Y$ was not generated from $X$, the result describes relative proximity but should not be interpreted as literal membership leakage.

We use the notation from the series:

- $\widetilde X$ is the complete preprocessed representation of the training dataset, whose rows are members, with dimensions $n_X\times\widetilde d$.
- $\widetilde Y$ is the complete preprocessed representation of the released dataset, with dimensions $n_Y\times\widetilde d$.
- $V$ is the independent real holdout that was not used to train the generator, with dimensions $n_V\times d$.
- $\widetilde V$ is its complete preprocessed representation, whose rows are non-members, with dimensions $n_V\times\widetilde d$.

This metric uses no classifier. It lies in $[0,1]$, and $0.5$ means that distance provides no membership-ranking signal.

## 2. Mixed-type record distance

Let $N$ be the numerical columns and $C$ the categorical columns of a complete representation. The total number of columns is $\widetilde d=|N|+|C|$. Numerical values are already on the shared scale.

A **candidate record** $r$ is a row whose membership status is being tested. It is either a member $x\in\widetilde X$ or a non-member $v\in\widetilde V$. A row $y\in\widetilde Y$ is always taken from the released dataset. Thus, $r$ and $y$ are not rows from the same table in this calculation. For a candidate $r$ and a released row $y$, define

$$
\delta(r,y)=\frac1{\widetilde d}\left[
\sum_{j\in N}|r_j-y_j|+
\sum_{j\in C}\mathbf1\{r_j\ne y_j\}
\right]
$$

Numerical columns contribute absolute differences. Categorical columns contribute $0$ when they match and $1$ when they differ. Thus, this is a mixed-type distance: its categorical part is Hamming distance, while its numerical part is scaled absolute distance.

The distance of candidate record $r$ to the released table is its distance to the closest row in $\widetilde Y$:

$$
\delta_Y(r)=\min_{y\in\widetilde Y}\delta(r,y)
$$

The subscript $Y$ means that the closest-row search is performed only within $\widetilde Y$. In particular,

$$
\delta_Y(x)=\min_{y\in\widetilde Y}\delta(x,y),\qquad x\in\widetilde X,
$$

and

$$
\delta_Y(v)=\min_{y\in\widetilde Y}\delta(v,y),\qquad v\in\widetilde V.
$$

A smaller value means that the candidate has a closer representative in the released dataset.

## 3. Membership score and AUC

The true membership label is

$$
L(r)=
\begin{cases}
1, & r=x\in\widetilde X \quad\text{(member)},\\
0, & r=v\in\widetilde V \quad\text{(non-member)}.
\end{cases}
$$

The quantity $\delta_Y(r)$ is a distance and remains nonnegative. AUROC, however, follows the convention that a larger score should indicate stronger evidence for the positive class, which here is membership ($L=1$). Because a smaller distance is stronger evidence of membership, define the attack score by reversing the distance ordering:

$$
s(r)=-\delta_Y(r)
$$

The value $s(r)$ is not a distance; it is only a ranking score. For example, if $\delta_Y(x)=0.10$ for a member and $\delta_Y(v)=0.40$ for a non-member, then $s(x)=-0.10>-0.40=s(v)$, so the member receives the higher attack score. Because the distances lie in $[0,1]$, the proximity score $1-\delta_Y(r)$ would produce exactly the same ranking and AUROC.

The metric is the [AUROC](7.1-auroc-for-binary-classification.ipynb) of $s$. For a randomly selected member $x$ and non-member $v$,

$$
s(x)>s(v)\quad\Longleftrightarrow\quad\delta_Y(x)<\delta_Y(v).
$$

Therefore,

$$
\mathrm{MIA}_{\mathrm{privacy}}(X,Y,V)=
P(\delta_Y(x)<\delta_Y(v))+\frac12P(\delta_Y(x)=\delta_Y(v))
$$

In a finite dataset, this is the average credit over every member–non-member pair: $1$ when the member is closer to $\widetilde Y$, $1/2$ for a tie, and $0$ when the non-member is closer.

AUC $0.5$ means that distance cannot rank members above non-members. AUC $1$ means every sampled member is closer to $Y$ than every sampled non-member.

In [2]:
#| label: dcr-membership

example = make_example_views()
X_tilde = example.X_tilde
Y_tilde = example.Y_tilde
V_tilde = example.V_tilde

details = distance_membership_details(
    X_tilde,
    V_tilde,
    Y_tilde,
    categorical_columns=METRIC_CATEGORICAL_COLUMNS,
)
MIA_privacy = distance_membership_auc(
    X_tilde,
    V_tilde,
    Y_tilde,
    categorical_columns=METRIC_CATEGORICAL_COLUMNS,
)

display(details.round(4))
display(details.groupby("group")["closest_distance"].agg(["count", "mean", "median"]).round(4))
print(f"Membership-inference AUC = {MIA_privacy:.6f}")

,group,label,closest_distance,attack_score
0,member,1.0,0.0131,-0.0131
1,member,1.0,0.2312,-0.2312
2,member,1.0,0.1780,-0.1780
3,member,1.0,0.0093,-0.0093
4,member,1.0,0.0177,-0.0177
5,member,1.0,0.1844,-0.1844
6,non-member,0.0,0.0075,-0.0075
7,non-member,0.0,0.3773,-0.3773
8,non-member,0.0,0.1862,-0.1862
9,non-member,0.0,0.2006,-0.2006


,count,mean,median
group,,,
member,6,0.1056,0.0978
non-member,10,0.1561,0.1816


Membership-inference AUC = 0.633333


## 4. Interpretation

The shared example gives an AUC above $0.5$, so members tend to lie somewhat closer to $Y$ than independent non-members do. With such small tables, treat the number only as a transparent calculation.

Unlike exact-match rate, this metric detects near copies. Its roles are directional: $X$ supplies members, $Y$ is the released table, and $V$ supplies non-members. Swapping them changes the question. Values below $0.5$ indicate reversed ranking rather than stronger membership leakage.

## Takeaway

Treat rows of $X$ as members and independent rows of $V$ as non-members. Measure every candidate's distance to its closest released row in $Y$, reverse the distance ordering with the attack score $s(r)=-\delta_Y(r)$, and report how well that score ranks members above non-members using AUROC.